In [9]:
import pandas as pd
import numpy as np
from google.colab import files

# ===================================================================
# 0. 원본 데이터 업로드 및 첫 열(Timestamp) 제거 -> Result부터 시작
# ===================================================================
uploaded = files.upload()
file_name = list(uploaded.keys())[0]
df = pd.read_excel(file_name, sheet_name='Data')

if 'Timestamp' in df.columns:
    df = df.drop(columns=['Timestamp'])

meta_cols = ['Result', 'Pass_Fail']
all_features = [c for c in df.columns if c not in meta_cols]

print(f"■ Timestamp 제거 후 크기: {df.shape[0]:,}행 × {df.shape[1]:,}열 (첫 컬럼: {df.columns[0]})")

# ===================================================================
# [1단계] 결측률 25% 초과 행·열 및 단일값(상수) 열 제거
# ===================================================================
# 1-1. 결측률 25% 초과 행(샘플) 제거
row_missing_rate = df[all_features].isna().mean(axis=1)
removed_rows_cnt = (row_missing_rate > 0.25).sum()

# 1-2. 결측률 25% 초과 열 또는 단일값(변동성 없는 상수) 열 제거
col_missing_rate = df[all_features].isna().mean()
valid_feature_cols = [
    c for c in all_features
    if (col_missing_rate[c] <= 0.25) and (df[c].dropna().nunique() > 1)
]
removed_cols_cnt = len(all_features) - len(valid_feature_cols)

df_step1 = df.loc[row_missing_rate <= 0.25, meta_cols + valid_feature_cols].reset_index(drop=True)

print(f"■ [1단계 완료] 제거된 행: {removed_rows_cnt}개, 제거된 열: {removed_cols_cnt}개 ➔ 잔여: {df_step1.shape[0]:,}행 × {df_step1.shape[1]:,}열\n")

# 타겟 컬럼 확인 (Pass / Fail 구분)
pass_mask = (df_step1['Pass_Fail'] == -1) if 'Pass_Fail' in df_step1.columns else (df_step1['Result'] == 'Pass')
fail_mask = ~pass_mask

df_processed = df_step1.copy()

summary_list = []
total_outliers_count = 0
total_missing_before = df_step1[valid_feature_cols].isna().sum().sum()

# ===================================================================
# [2단계] Pass 전용 6σ 이상치 판별 및 상·하한선 대체 (Clipping 우선 수행)
# ===================================================================
for col in valid_feature_cols:
    pass_vals_raw = df_step1.loc[pass_mask, col].dropna()
    fail_vals_raw = df_step1.loc[fail_mask, col].dropna()

    pass_count = len(pass_vals_raw)
    fail_count = len(fail_vals_raw)

    Q1 = pass_vals_raw.quantile(0.25)
    Q3 = pass_vals_raw.quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 3.0 * IQR
    upper_bound = Q3 + 3.0 * IQR

    pass_outlier_mask = pass_mask & ((df_step1[col] < lower_bound) | (df_step1[col] > upper_bound))
    outlier_cnt = pass_outlier_mask.sum()
    total_outliers_count += outlier_cnt

    # Pass 데이터에 한해 상·하한선 대체 (Fail 이상치는 원본 보존)
    df_processed.loc[pass_mask & (df_processed[col] > upper_bound), col] = upper_bound
    df_processed.loc[pass_mask & (df_processed[col] < lower_bound), col] = lower_bound

    # ===============================================================
    # [3단계] 이상치가 정제된 Pass 데이터의 중앙값으로 잔여 결측치(NaN) 대치
    # ===============================================================
    pass_vals_clipped = df_processed.loc[pass_mask, col].dropna()
    med_val = pass_vals_clipped.median() if len(pass_vals_clipped) > 0 else df_processed[col].median()

    # Pass와 Fail 모두 Pass 기준 중앙값으로 통일 대치 (데이터 누수 방지)
    df_processed[col] = df_processed[col].fillna(med_val)

    summary_list.append({
        'Feature': col,
        'Pass_데이터수': pass_count,
        'Fail_데이터수': fail_count,
        'Pass_이상치수': outlier_cnt,
        '대치_중앙값': round(med_val, 4)
    })

# ===================================================================
# 4. 전처리 결과 요약 출력 및 최종 엑셀 파일 다운로드
# ===================================================================
summary_df = pd.DataFrame(summary_list)

print("=" * 75)
print("■ Feature별 데이터 수, Pass 이상치(Clipping) 수 및 대치 중앙값 요약")
print("=" * 75)
print(summary_df.to_string(index=False))

print("\n" + "=" * 75)
print(f"★ [1단계] 정제 후 데이터 크기            : {df_processed.shape[0]:,}행 × {df_processed.shape[1]:,}열")
print(f"★ [2단계] 처리된 Pass 그룹 내 이상치 개수 : {total_outliers_count:,} 개 (Clipping 완료)")
print(f"★ [3단계] 대치 전 총 결측치(NaN) 개수     : {total_missing_before:,} 개")
print(f"★ [3단계] 대치 후 잔여 결측치(NaN) 개수   : {df_processed[valid_feature_cols].isna().sum().sum():,} 개 (완벽 제거)")
print("=" * 75)

output_filename = 'secom_25%_중앙값대치_이상치처리.xlsx'
df_processed.to_excel(output_filename, index=False)
print(f"\n[완료] '{output_filename}' 파일 생성이 완료되어 다운로드를 시작합니다.")
files.download(output_filename)

Saving secom_data.xlsx to secom_data (2).xlsx
■ Timestamp 제거 후 크기: 1,567행 × 592열 (첫 컬럼: Result)
■ [1단계 완료] 제거된 행: 2개, 제거된 열: 148개 ➔ 잔여: 1,565행 × 444열

■ Feature별 데이터 수, Pass 이상치(Clipping) 수 및 대치 중앙값 요약
    Feature  Pass_데이터수  Fail_데이터수  Pass_이상치수     대치_중앙값
  Feature_1       1455        104          2  3012.9800
  Feature_2       1455        103         10  2499.3500
  Feature_3       1447        104          7  2201.0667
  Feature_4       1447        104         12  1285.2144
  Feature_5       1447        104         11     1.3168
  Feature_7       1447        104          3   101.5122
  Feature_8       1452        104          8     0.1224
  Feature_9       1461        104          0     1.4611
 Feature_10       1461        104          1    -0.0013
 Feature_11       1461        104          1     0.0003
 Feature_12       1461        104          3     0.9659
 Feature_13       1461        104          3   199.5093
 Feature_15       1460        104          0     8.9820
 Feature_16   

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>